# 9 · Overlays — your own layers on the road map

An `Overlay` draws any geometry with the roads: **zone fills under** the network, **POI markers or route lines over** it. Each overlay gets a click popup from the fields you list and a spot in the in-map **Layers** toggle. Kind (fill / line / circle) is auto-detected from the geometry.

**Roads vs. overlays:** the road layer is the *subject* — the full cartographic engine (widths, casing, labels, arrows, grade separation, data-driven colour). Overlays are *annotations*: one flat paint each, a popup, a Layers-toggle checkbox. They deliberately never grow casings or labels, so the roads stay visually dominant. A second linear network that deserves road-grade treatment should be rendered *as* the main layer (custom `highway_col` + palette), not as an overlay.


In [1]:
import pathlib
import geopandas as gpd
import numpy as np
from shapely.geometry import box

import roadstyle as rs
from roadstyle import Overlay

edges = gpd.read_file(pathlib.Path("data") / "sodermalm_edges.gpkg")
print(f"{len(edges):,} edges")

4,563 edges


ERROR 1: PROJ: proj_create_from_database: Open of /home/kaveh/miniconda3/envs/roadstyle/share/proj failed


## Fabricate some layers from the sample data

Districts (a coarse grid over the island), "sensors" (midpoints of random service edges), and a route (every edge of Hornsgatan) — stand-ins for your zones, measurement sites, and paths.


In [2]:
minx, miny, maxx, maxy = edges.total_bounds
xs, ys = np.linspace(minx, maxx, 4), np.linspace(miny, maxy, 3)
districts = gpd.GeoDataFrame(
    {"district": [f"D{i}" for i in range(6)],
     "population": (np.arange(6) + 1) * 3200},
    geometry=[box(xs[i], ys[j], xs[i + 1], ys[j + 1])
              for j in range(2) for i in range(3)], crs=4326)

rng = np.random.default_rng(0)
sites = edges[edges.highway == "service"].sample(25, random_state=0)
sensors = gpd.GeoDataFrame(
    {"sensor_id": [f"S{i:03d}" for i in range(25)],
     "status": rng.choice(["ok", "warn"], 25, p=[0.8, 0.2])},
    geometry=sites.geometry.interpolate(0.5, normalized=True).values, crs=4326)

route = edges[edges.name == "Hornsgatan"]
len(districts), len(sensors), len(route)

/tmp/ipykernel_1228529/3490879892.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  geometry=sites.geometry.interpolate(0.5, normalized=True).values, crs=4326)


(6, 25, 75)

## Explicit per-overlay styling

Each overlay carries its own colour and popup; `placement` picks the stacking side.


In [3]:
rs.render_edges(
    edges, basemap="dark_matter", compress=True,
    overlays=[
        Overlay(districts, placement="under", color="#2d6cdf", opacity=0.18,
                label="Districts", popup=["district", "population"]),
        Overlay(route, placement="over", color="#ff2d78", width=5,
                label="Hornsgatan", popup=["name", "maxspeed_kmh"]),
        Overlay(sensors, placement="over", color="#ffd166",
                label="Sensors", popup=["sensor_id", "status"]),
    ],
)

## Styling MANY overlays at once: the `overlays` settings block

Overlay *defaults* (colour, circle radius, widths, per-kind opacities, circle stroke) live in the settings file like every other style default. When several overlays don't set their own values, one settings block styles them all — here per-call via `settings=`, equally valid in a `roadstyle.json`:

```json
{"config": {"overlays": {"color": "#ff6b6b", "radius": 9, ...}}}
```

**Precedence:** a per-`Overlay` value always beats the setting — the districts below stay blue while every default-styled overlay follows the settings block.


In [4]:
ok = sensors[sensors.status == "ok"]
warn = sensors[sensors.status == "warn"]

rs.render_edges(
    edges, basemap="dark_matter", compress=True,
    overlays=[
        Overlay(districts, placement="under", color="#2d6cdf", opacity=0.18,
                label="Districts", popup=["district", "population"]),   # explicit -> wins
        Overlay(ok, label="Sensors OK", popup=["sensor_id"]),            # settings-styled
        Overlay(warn, label="Sensors WARN", popup=["sensor_id"]),        # settings-styled
        Overlay(route, label="Hornsgatan", popup=["name"]),              # settings-styled
    ],
    settings={"config": {"overlays": {
        "color": "#ff6b6b", "radius": 9, "width": 4,
        "circle_stroke": "#000000", "circle_opacity": 0.95,
    }}},
)

Toggle the layers from the **Layers** control (top-right), click any zone / sensor / route segment for its popup — and note the roads' own hover/click still work underneath. In real use the settings block usually lives in your `roadstyle.json`, so every map in a project shares the same overlay look with zero per-call styling.


## Querying an overlay from JavaScript

Overlays are queryable tables too: every `rs*` id-set verb takes an **optional layer argument** — omitted it means the roads, an overlay's `label` (or index) means that overlay. In any saved page:

```js
const bad = rsQuery(p => p.status === 'offline', "Sensors");   // the overlay as the table
rsHighlight(bad, "Sensors");            // glow those sensors
rsColor(bad, "#ff1744", "Sensors");     // paint them red (reset: rsColor(null, null, "Sensors"))
rsFilter(bad, "Sensors");               // show only them   (reset: rsFilter(null, "Sensors"))
rsGetProps(bad, "Sensors");             // their rows, for a table
rsFocus(bad, null, "Sensors");          // fly the camera to them
```

Each layer is its **own id space** — ids from a roads query and an overlay query are different things; never mix them. See notebook 03 for the full JS API and the `ui/` folder for a ready-made dashboard built on it.